# Full Workflow: Quantifying Topological Defect Influence

**Project**: PROJ-209-quantifying-the-influence-of-topological
**Task**: T033 - Runtime Optimization for CI (≤6h, ≤7GB RAM)

This notebook orchestrates the end-to-end pipeline:
1. Data Acquisition (Real or Synthetic Fallback)
2. Data Processing & Normalization
3. Model Training (Random Forest) & Collinearity Handling
4. Inference (Permutation Importance, FDR)
5. Validation & Sensitivity Analysis

**Runtime Constraint**: Optimized for GitHub Actions Free Tier (CPU, 7GB RAM, 6h limit).

In [ ]:
import os
import sys
import time
import json
import logging
from pathlib import Path

# Add project root to path for imports
PROJECT_ROOT = Path(__file__).parent.parent
CODE_DIR = PROJECT_ROOT / "code"
sys.path.insert(0, str(CODE_DIR))

# Configure logging for runtime monitoring
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Import pipeline modules
from 01_data_acquisition import run_acquisition
from 02_data_processing import process_data, update_state_with_checksums
from 03_modeling import run_collinearity_analysis, train_random_forest_models, save_metrics
from 04_inference import run_inference_analysis
from 05_validation import run_validation_analysis

## Step 1: Data Acquisition
Downloads pristine structures and attempts to fetch defect data. Falls back to synthetic generation if real data is unavailable.

In [ ]:
logger.info("Starting Step 1: Data Acquisition")
start_time = time.time()

try:
    # Run acquisition (handles API retries and synthetic fallback)
    run_acquisition()
    acquisition_time = time.time() - start_time
    logger.info(f"Step 1 completed in {acquisition_time:.2f}s")
except Exception as e:
    logger.critical(f"Data acquisition failed: {e}")
    raise

## Step 2: Data Processing
Normalizes targets, encodes defect types, and handles missing references.

In [ ]:
logger.info("Starting Step 2: Data Processing")
start_time = time.time()

try:
    # Process data and generate features/targets
    process_data()
    
    # Update state with checksums
    update_state_with_checksums()
    
    processing_time = time.time() - start_time
    logger.info(f"Step 2 completed in {processing_time:.2f}s")
except Exception as e:
    logger.critical(f"Data processing failed: {e}")
    raise

## Step 3: Modeling
Handles collinearity, trains Random Forest models, and evaluates baselines.

In [ ]:
logger.info("Starting Step 3: Modeling")
start_time = time.time()

try:
    # Analyze and handle collinearity (Ridge/Feature Exclusion)
    run_collinearity_analysis()
    
    # Train RF models and baselines
    models, metrics = train_random_forest_models()
    
    # Save models and metrics
    save_models(models)
    save_metrics(metrics)
    
    modeling_time = time.time() - start_time
    logger.info(f"Step 3 completed in {modeling_time:.2f}s")
except Exception as e:
    logger.critical(f"Modeling failed: {e}")
    raise

## Step 4: Inference
Computes permutation importance, applies FDR control, and runs sensitivity analysis.

In [ ]:
logger.info("Starting Step 4: Inference")
start_time = time.time()

try:
    # Run full inference pipeline
    inference_results = run_inference_analysis()
    
    inference_time = time.time() - start_time
    logger.info(f"Step 4 completed in {inference_time:.2f}s")
except Exception as e:
    logger.critical(f"Inference failed: {e}")
    raise

## Step 5: Validation
Performs stability analysis, checks external data, and generates the final report.

In [ ]:
logger.info("Starting Step 5: Validation")
start_time = time.time()

try:
    # Run validation analysis and generate report
    validation_results = run_validation_analysis()
    
    validation_time = time.time() - start_time
    logger.info(f"Step 5 completed in {validation_time:.2f}s")
except Exception as e:
    logger.critical(f"Validation failed: {e}")
    raise

## Summary & Runtime Check
Verifies the workflow completed within the 6-hour limit.

In [ ]:
total_time = time.time() - start_time
hours = total_time / 3600

print(f"\n{'='*50}")
print(f"WORKFLOW COMPLETED")
print(f"{'='*50}")
print(f"Total Runtime: {hours:.2f} hours ({total_time:.2f} seconds)")
print(f"Constraint: 6.00 hours (21600 seconds)")

if hours <= 6.0:
    print("✅ PASSED: Runtime within limit.")
else:
    print("❌ FAILED: Runtime exceeded 6-hour limit.")
    
print(f"{'='*50}")

# Output summary of key artifacts
artifacts = [
    "data/raw/pristine_structures.csv",
    "data/raw/defect_dataset_2022.csv",
    "data/processed/features.csv",
    "data/processed/targets.csv",
    "data/models/random_forest_models.pkl",
    "data/validation/Validation_Report.json"
]

print("\nGenerated Artifacts:")
for art in artifacts:
    p = Path(art)
    if p.exists():
        print(f"  ✅ {art} ({p.stat().st_size} bytes)")
    else:
        print(f"  ❌ {art} (Missing)")